In [ ]:
EXTRACT_USER_INFO_PROMPT = """
You are an expert user-information extraction, validation, and merging system.

Your task is to produce the FINAL UserInformation by using ONLY information
that is explicitly present in one of these three input sources:

1. EXISTING USER INFORMATION
2. RESUME
3. USER INSTRUCTION

Your job is to extract, validate, and merge information from those sources.

============================================================
CRITICAL DATA PROVENANCE RULE
============================================================

THIS RULE IS ABSOLUTE:

Every value in the final output MUST come directly from the provided
EXISTING USER INFORMATION, RESUME, or USER INSTRUCTION.

NEVER invent, infer, guess, autocomplete, hallucinate, recall, or generate
a value that does not appear in those input sources.

A value must not be added merely because it looks plausible.

If a value does not appear in the input sources, it MUST NOT appear in
the final output.

IMPORTANT:

- Text appearing inside THIS PROMPT is NOT user data.
- Examples appearing inside THIS PROMPT are NOT user data.
- Never copy values from examples in this prompt into the final output.
- Never use information from your general knowledge or prior knowledge.
- Never assume a likely email address, phone number, URL, name, company,
  or any other personal information.
- Never complete, correct, or "improve" a partially provided value unless
  the complete value is explicitly present in one of the three input sources.

Think of the three sections below as the ONLY database you are allowed
to use:

EXISTING USER INFORMATION
RESUME
USER INSTRUCTION

Nothing else is a valid source of user information.


============================================================
EXISTING USER INFORMATION
============================================================

{existing_user_info}


============================================================
RESUME
============================================================

{resume_content}


============================================================
OPTIONAL USER INSTRUCTION
============================================================

{user_instruction}


============================================================
SOURCE PRIORITY AND MERGING
============================================================

Use the following rules:

1. Keep existing valid information from EXISTING USER INFORMATION.

2. Extract additional information from the RESUME only when the information
   is explicitly present there and clearly belongs to the user.

3. Use USER INSTRUCTION when it explicitly provides a value or explicitly
   requests a change.

4. If the same field contains multiple valid values explicitly present in
   the allowed sources, keep the unique values.

5. Never create a new value from a combination of other values.

6. Never infer a value from a person's name, username, domain, company,
   university, GitHub username, LinkedIn username, or any other information.

7. Never assume that an email address, phone number, URL, or other personal
   information belongs to the user unless the source clearly supports it.

8. If a value is absent from all three input sources, return an empty string
   for that field.

9. If a value appears only in this prompt, including examples or
   instructions, it MUST NOT be returned.

10. Preserve the value as written in the source whenever possible.
    Do not fabricate missing characters or information.


============================================================
FIELD VALIDATION
============================================================

Every returned value must satisfy BOTH conditions:

A. It is explicitly present in one of the allowed input sources.
B. It belongs to the correct field and clearly belongs to the user.

------------------------------------------------------------
full_name
------------------------------------------------------------

Return only the person's actual full name.

Do not include:

- job titles
- company names
- descriptions
- email addresses
- phone numbers
- URLs
- university names

Do not infer or construct a name from an email address or username.

------------------------------------------------------------
phone
------------------------------------------------------------

Extract phone number(s) that are explicitly present in the allowed input
sources and are associated with the user/candidate.

A phone number should be considered the user's phone number when it appears
in a personal contact/header section associated with the candidate, for
example near:

- the candidate's name
- email address
- LinkedIn URL
- GitHub URL
- portfolio URL
- contact information
- resume header

If a phone number appears in such a personal contact section, extract it.

If multiple phone numbers are explicitly associated with the candidate,
return all unique phone numbers as a comma-separated string.

Do NOT:

- invent a phone number
- guess missing digits
- infer a phone number from other information
- modify a phone number into a different number
- extract company/recruiter phone numbers
- extract unrelated numbers such as application IDs, admission numbers,
  registration numbers, postal codes, dates, years, or other numeric values

IMPORTANT:

A number does NOT need to be labelled exactly as "Phone" to be extracted.

For example, if a resume header contains the candidate's name followed by
an email address, LinkedIn/GitHub information, and a 10-digit telephone
number, that telephone number should be treated as the candidate's phone
number when the surrounding context clearly indicates it is contact
information.

The phone number must still be explicitly present in the allowed source.
Never generate a phone number that is not present in the source.

------------------------------------------------------------
email
------------------------------------------------------------

Return only email address(es) that are explicitly present in the allowed
input sources and clearly belong to the user.

If multiple valid email addresses exist, return them as a comma-separated
string.

ABSOLUTE EMAIL RULE:

Before returning an email address, verify that the EXACT email address
appears somewhere in EXISTING USER INFORMATION, RESUME, or USER INSTRUCTION.

If the exact email address does not appear in one of those sources,
DO NOT RETURN IT.

Never:

- invent an email address
- infer an email from the person's name
- infer an email from a GitHub username
- infer an email from a domain
- autocomplete an email
- use an email from an example in this prompt
- use an email from general knowledge
- assume a second email exists

------------------------------------------------------------
linkedin_url
------------------------------------------------------------

Return only the user's LinkedIn profile URL explicitly present in the
allowed input sources.

The URL must actually be a LinkedIn profile URL.

Never construct a LinkedIn URL from the person's name.

------------------------------------------------------------
github_url
------------------------------------------------------------

Return only the user's GitHub profile URL explicitly present in the
allowed input sources.

The URL must actually be a GitHub profile URL.

Never construct a GitHub URL from the person's name or username unless
the complete URL is explicitly present in an allowed input source.

------------------------------------------------------------
portfolio_url
------------------------------------------------------------

Return only the user's personal portfolio/personal website URL explicitly
present in the allowed input sources.

Do not infer a portfolio URL.

Do not construct a portfolio URL from the person's name.

Do not use:

- LinkedIn
- GitHub
- company websites
- university websites
- unrelated websites


============================================================
USER INSTRUCTION
============================================================

The user instruction is optional.

If it is empty, ignore it.

If it provides a new value, that value may be used ONLY if the value is
explicitly written in the user instruction.

For example, if the user instruction explicitly contains a new email
address, that email may be considered.

However:

- Never invent a value based on the instruction.
- Never complete a partial value.
- Never infer missing information.
- Never copy values from examples in this prompt.
- Never treat instructions or examples from THIS PROMPT as user data.

If the user instruction explicitly requests changing a field, follow that
instruction only when a clear value is provided.

If the instruction says something like "use my new email" but does not
actually provide the email address, do not invent one.


============================================================
EXACT SOURCE VERIFICATION
============================================================

Before returning EACH non-empty field, perform this verification:

STEP 1:
Identify the exact value you intend to return.

STEP 2:
Find that exact value in one of:

- EXISTING USER INFORMATION
- RESUME
- USER INSTRUCTION

STEP 3:
If the exact value cannot be found in one of those sources, DELETE it.

STEP 4:
Verify that the value belongs to the user.

STEP 5:
Verify that the value belongs to the correct field.

Only values that pass ALL five steps may be returned.


============================================================
NO HALLUCINATION RULE
============================================================

When uncertain, return an empty string.

An empty field is ALWAYS preferable to a guessed value.

Never attempt to be helpful by generating missing personal information.

Never use common patterns to guess personal information.

For example, do NOT infer:

- an email from a person's name
- an email from a GitHub username
- a portfolio from a person's name
- a LinkedIn URL from a person's name
- a phone number from another phone number
- missing digits
- missing domain names
- alternative versions of an email address

ONLY extract information that is actually present in the allowed sources.


============================================================
DUPLICATE HANDLING
============================================================

Do not return duplicate values.

For fields that support multiple values, return unique valid values as a
comma-separated string.

Do not create additional values while deduplicating.


============================================================
VALUE FORMAT
============================================================

Every field must contain ONLY the actual value.

Never include:

- explanations
- sentences
- labels
- comments
- field names
- prefixes
- suffixes
- reasoning

Return only the structured UserInformation output.


============================================================
FINAL VALIDATION
============================================================

Before returning the final result, verify ALL of the following:

1. Every non-empty value appears explicitly in EXISTING USER INFORMATION,
   RESUME, or USER INSTRUCTION.

2. No value came from this prompt's examples.

3. No value was inferred or guessed.

4. No value was generated from general knowledge.

5. Every email address appears EXACTLY in an allowed input source.

6. Every phone number appears explicitly in an allowed input source.

7. Every URL appears explicitly in an allowed input source.

8. Every value actually belongs to the user.

9. Every value belongs to the correct field.

10. Duplicate values have been removed.

11. If there is any uncertainty, the field is an empty string.

Return ONLY the structured UserInformation output.

{format_instructions}
"""

In [2]:
from pydantic import BaseModel, Field


class UserInformation(BaseModel):
    full_name: str = Field(
        default="",
        description="The full name of the person."
    )

    phone: str = Field(
        default="",
        description=(
            "All phone numbers belonging to the person, "
            "as a comma-separated string. Empty string if not found."
        )
    )

    linkedin_url: str = Field(
        default="",
        description="The person's LinkedIn profile URL. Empty string if not found."
    )

    github_url: str = Field(
        default="",
        description="The person's GitHub profile URL. Empty string if not found."
    )

    portfolio_url: str = Field(
        default="",
        description="The person's personal portfolio URL. Empty string if not found."
    )

    email: str = Field(
        default="",
        description=(
            "All email addresses belonging to the person, "
            "as a comma-separated string. Empty string if not found."
        )
    )

In [ ]:
import json
import os

from dotenv import load_dotenv

from langchain_huggingface import (
    HuggingFaceEndpoint,
    ChatHuggingFace,
)

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_classic.output_parsers import OutputFixingParser

from .extract_user_info_prompt import EXTRACT_USER_INFO_PROMPT
from .extract_user_info_schema import UserInformation


load_dotenv()


llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    huggingfacehub_api_token=os.environ["HUGGINGFACEHUB_API_TOKEN1"],
    max_new_tokens=1024,
    temperature=0.1,
)

model = ChatHuggingFace(llm=llm)


parser = PydanticOutputParser(
    pydantic_object=UserInformation
)


fixing_parser = OutputFixingParser.from_llm(
    parser=parser,
    llm=model,
    max_retries=2,
)


prompt = ChatPromptTemplate.from_template(
    EXTRACT_USER_INFO_PROMPT
)


extract_user_info_chain = prompt | model | fixing_parser


def _normalize_existing_user_info(
    existing_user_info: UserInformation | dict | str | None,
) -> str:
    """
    Convert existing user information into a predictable JSON string
    before sending it to the LLM.
    """

    if existing_user_info is None:
        return "{}"

    if isinstance(existing_user_info, UserInformation):
        return existing_user_info.model_dump_json(indent=2)

    if isinstance(existing_user_info, dict):
        return json.dumps(
            existing_user_info,
            ensure_ascii=False,
            indent=2,
        )

    if isinstance(existing_user_info, str):
        existing_user_info = existing_user_info.strip()

        if not existing_user_info:
            return "{}"

        # If the caller already supplied JSON, preserve it.
        try:
            parsed = json.loads(existing_user_info)

            return json.dumps(
                parsed,
                ensure_ascii=False,
                indent=2,
            )

        except json.JSONDecodeError:
            # Keep backward compatibility if somebody passes
            # plain text instead of JSON.
            return existing_user_info

    raise TypeError(
        "existing_user_info must be UserInformation, dict, str, or None"
    )


def extract_user_information(
    resume_content: str,
    existing_user_info: UserInformation | dict | str | None = None,
    user_instruction: str = "",
) -> UserInformation:

    if not resume_content or not resume_content.strip():
        raise ValueError("resume_content must not be empty")

    existing_user_info_content = _normalize_existing_user_info(
        existing_user_info
    )

    result = extract_user_info_chain.invoke(
        {
            "resume_content": resume_content.strip(),

            "existing_user_info": existing_user_info_content,

            "user_instruction": (
                user_instruction.strip()
                if user_instruction
                else ""
            ),

            "format_instructions": parser.get_format_instructions(),
        }
    )

    return result

In [5]:
resume = """
# **Anirban Das** 

**Adm. No.** 23JE0104 � 6290375587 � My <u>portfolio website-https://anirban-das-portfolio.vercel.app/</u>
� dasanirban268@gmail.com � <u>linkedin.com/in/anirbandas</u> � <u>github.com/anirban2005143a</u> 

## Education 

### **Indian Institute of Technology (Indian School of Mines), Dhanbad** 

_Bachelor of Technology in Computer Science and Engineering (GPA: 8.29 / 10.00)_ 

Expected May 2027 _Dhanbad, Jharkhand_ 

- **Relevant Coursework:** Data Structures and Algorithms (C++), Database Management System, Compiler Design, Computer Organization , Computer Architecture, Operating Systems. 

## Experience 

### **Google** _|_ **_Software Engineer Intern_** 

May 2026 – July 2026 

- Engineered a new linting rule and automated batch validation pipeline in Java and TypeScript to detect structural errors across cloud contract templates. 

- Enhanced core template management interfaces by developing a responsive document comparison dialog box, a read/write mode selector for gDoc, and an embedded AI assistant chat widget. 

- Contributed 7,350+ lines of production code across 22 peer-reviewed changelists and created analytical dashboards for contract monitoring and template health tracking. 

## Projects 

**<u>Code Fusion</u>** _| React.js, Flask, Express.js, MongoDB, Tailwind CSS |_ _<u>GitHub</u> | Deployed Project_ 

- An online code editor supporting real-time collaboration, multiple languages, and customizable themes. 

- Enables multiple developers to collaborate in real-time with live cursor tracking and integrated in-app chat for seamless communication. 

- Integrates auto-completion, real-time syntax error detection, and persistent code-saving functionality, while offering various themes, multi-language support, and a collapsible sidebar for efficient file management. 

**<u>JobPilot</u>** _| Python (FastAPI), TypeScript (Next.js), Langchain, WebSockets |_ _<u>GitHub</u> |_ _<u>Video</u>_ 

- Developed a full-stack automation system using **FastAPI** , **Next.js** , and **HuggingFace LLMs** to automate resume extraction and job matching, reducing manual application time by **80%** . 

- Architected a scalable background worker system with **asynchronous processing** , **WebSockets** for real-time updates, and **FileLock** synchronization to manage concurrent multi-user job lifecycles. 

- Built an **LLM-powered** pipeline using **prompt engineering** for job ranking and categorization, featuring a manual review workflow and a local mock portal for safe system validation. 

**NoteBridge** _| React.js, Express.js, MongoDB, Bootstrap |_ _<u>GitHub</u> | Deployed Project_ 

- A **feature-rich note-taking and sharing platform** that enables **structured organization** through folders and facilitates **controlled file sharing** . 

- Enables **interactive engagement** through features like **likes** , **comments** , and **shares** . 

- Provides a **comprehensive profile page** displaying total posts, followers, following, and an **organized archive of past posts** for easy access and engagement. 

## Technical Skills 

**AI/ML & Agents** : LangChain, LangGraph, TensorFlow, Keras, Deep Learning, ANN, CNN, LSTM. **Technologies** : Node.js, FastAPI, Express.js, Docker, Next.js, React.js, Tailwind CSS, Three.js, GSAP. **Database & Cloud** : MongoDB, PostgreSQL, Vector Databases. 

## Achievements 

- Secured **4th** rank at **HaXplore** _|_ **CodeFest’25** , organized by **IIT BHU!** 

- **Winner** - of Winter Of Code 6.O (in Web Development Division) a one-month long hackathon conducted by **CyberLabs** , IIT(ISM) Dhanbad. _| Deployed Project_ 

## Social Engagements 

- Member of CyberLabs -Tech society of IIT ISM Dhanbad 

- Member of Aquatics Team - Swimming Team of IIT ISM Dhanbad. 

- Represented IIT Dhanbad at the 37th INTER IIT AQUATICS MEET 2023 held at IIT Gandhinagar and secured **4th place** in 200m Individual Medley . 


"""

In [6]:
existing_user_info = UserInformation(
    full_name="Anirban Das",
    phone="9876543210",
    email="anirban@gmail.com",
    linkedin_url="linkedin.com/in/anirban",
    github_url="github.com/anirban",
    portfolio_url="https://anirban.dev",
)

In [8]:
result = extract_user_information(
    resume_content=resume,
    existing_user_info=existing_user_info,
)

print(result.model_dump())

{'full_name': 'Anirban Das', 'phone': '9876543210, 6290375587', 'linkedin_url': 'https://linkedin.com/in/anirbandas', 'github_url': 'https://github.com/anirban2005143a', 'portfolio_url': 'https://anirban-das-portfolio.vercel.app/', 'email': 'anirban@gmail.com, dasanirban268@gmail.com'}
